<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0">

# Mean-Variance Portfolio Class

Without doubt, the Markowitz (1952) **mean-variance portfolio theory** is a cornerstone of modern financial theory. This section illustrates the use of the `mean_variance_portfolio` class to implement this approach.

In [ ]:
!git clone https://github.com/tpq-classes/computational_finance_dx.git
import sys
sys.path.append('computational_finance_dx')

!git clone https://github.com/yhilpisch/dx.git
sys.path.append('dx')


In [ ]:
from dx import *
from pylab import plt
plt.style.use('seaborn-v0_8')

## Market Environment and Portfolio Object

We start by instantiating a `market environment` object which in particular contains a list of **ticker symbols** in which we are interested in.

In [ ]:
ma = market_environment('ma', dt.date(2010, 1, 1))
ma.add_list('symbols', ['AAPL.O', 'INTC.O', 'MSFT.O', 'GS.N'])
ma.add_constant('source', 'google')  # not relevant anymore
ma.add_constant('final date', dt.date(2014, 3, 1))

Using pandas under the hood, the class **retrieves historial stock price data** from a given data source.

In [ ]:
%%time
port = mean_variance_portfolio('am_tech_stocks', ma)
  # instantiates the portfolio class
  # and retrieves all the time series data needed

In [ ]:
port.get_available_symbols()

## Basic Statistics

Since no **portfolio weights** have been provided, the class defaults to equal weights.

In [ ]:
port.get_weights()
  # defaults to equal weights

Given these weights you can calculate the **portfolio return** via the method `get_portfolio_return`.

In [ ]:
port.get_portfolio_return()
  # expected (= historical mean) return

Analogously, you can call `get_portfolio_variance` to get the historical **portfolio variance**.

In [ ]:
port.get_portfolio_variance()
  # expected (= historical) variance

The class also has a neatly printable `string` representation.

In [ ]:
print(port)
  # ret. con. is "return contribution"
  # given the mean return and the weight
  # of the security

## Setting Weights

Via the method `set_weights` the weights of the single portfolio components can be adjusted.

In [ ]:
port.set_weights([0.6, 0.2, 0.1, 0.1])

In [ ]:
print(port)

You cal also easily **check results for different weights** with changing the attribute values of an object.

In [ ]:
port.test_weights([0.6, 0.1, 0.2, 0.1])
    # returns av. return + vol + Sharp ratio
    # without setting new weights

Let us implement a **Monte Carlo simulation** over potential portfolio weights.

In [ ]:
# Monte Carlo simulation of portfolio compositions
rets = []
vols = []

for w in range(500):
    weights = np.random.random(4)
    weights /= sum(weights)
    r, v, sr = port.test_weights(weights)
    rets.append(r)
    vols.append(v)

rets = np.array(rets)
vols = np.array(vols)

And the simulation results **visualized**.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.figure(figsize=(10, 6))
plt.scatter(vols, rets, c=rets / vols, marker='o', cmap='coolwarm')
plt.xlabel('expected volatility')
plt.ylabel('expected return')
plt.colorbar(label='Sharpe ratio');

## Optimizing Portfolio Composition

One of the major application areas of the mean-variance portfolio theory and therewith of this DX Analytics class it the **optimization of the portfolio composition**. Different target functions can be used to this end.

### Return

The first target function might be the **portfolio return**.

In [ ]:
port.optimize('Return')
  # maximizes expected return of portfolio
  # no volatility constraint

In [ ]:
print(port)

Instead of maximizing the portfolio return without any constraints, you can also set a (sensible/possible) **maximum target volatility** level as a constraint. Both, in an **exact sense** ("equality constraint") ...

In [ ]:
port.optimize('Return', constraint=0.2, constraint_type='Exact')
  # interpretes volatility constraint as equality

In [ ]:
print(port)

... or just a an **upper bound** ("inequality constraint").

In [ ]:
port.optimize('Return', constraint=0.22, constraint_type='Bound')
  # interpretes volatility constraint as inequality (upper bound)

In [ ]:
print(port)

### Risk

The class also allows you to minimize **portfolio risk**.

In [ ]:
port.optimize('Vol')
  # minimizes expected volatility of portfolio
  # no return constraint

In [ ]:
print(port)

And, as before, to set **constraints** (in this case) for the target return level.

In [ ]:
port.optimize('Vol', constraint=0.175, constraint_type='Exact')
  # interpretes return constraint as equality

In [ ]:
print(port)

In [ ]:
port.optimize('Vol', constraint=0.15, constraint_type='Bound')
  # interpretes return constraint as inequality (upper bound)

In [ ]:
print(port)

### Sharpe Ratio

Often, the target of the portfolio optimization efforts is the so called **Sharpe ratio**. The `mean_variance_portfolio` class of DX Analytics assumes a **risk-free rate of zero** in this context.

In [ ]:
port.optimize('Sharpe')
  # maximize Sharpe ratio

In [ ]:
print(port)

## Efficient Frontier

Another application area is to derive the **efficient frontier** in the mean-variance space. These are all these portfolios for which there is **no portfolio with both lower risk and higher return**. The method `get_efficient_frontier` yields the desired results.

In [ ]:
%%time
evols, erets = port.get_efficient_frontier(100)
  # 100 points of the effient frontier

The plot with the **random and efficient portfolios**.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(vols, rets, c=rets / vols, marker='o')
plt.scatter(evols, erets, c=erets / evols, marker='o', cmap='coolwarm')
plt.xlabel('expected volatility')
plt.ylabel('expected return')
plt.colorbar(label='Sharpe ratio');

## Capital Market Line

The **capital market line** is another key element of the mean-variance portfolio approach representing all those risk-return combinations (in mean-variance space) that are possible to form from a **risk-less money market account** and **the market portfolio** (or another appropriate substitute efficient portfolio).

In [ ]:
%%time
cml, optv, optr = port.get_capital_market_line(riskless_asset=0.05)
  # capital market line for efficient frontier and risk-less short rate

In [ ]:
cml  # lambda function for capital market line

The following plot illustrates that the capital market line has an ordinate value equal to the **risk-free rate** (the risk-less return of the money market account) and is tangent to the **efficient frontier**.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(evols, erets, lw=2.0, label='efficient frontier')
plt.plot((0, 0.4), (cml(0), cml(0.4)), lw=2.0, label='capital market line')
plt.plot(optv, optr, 'r*', markersize=10, label='optimal portfolio')
plt.legend(loc=0)
plt.ylim(0)
plt.xlabel('expected volatility')
plt.ylabel('expected return');

Portfolio return and risk of the efficient portfolio used are:

In [ ]:
optr

In [ ]:
optv

The **portfolio composition** can be derived as follows.

In [ ]:
port.optimize('Vol', constraint=optr, constraint_type='Exact')

In [ ]:
print(port)

## More Assets

As a larger, more realistic example, consider a larger set of assets.

In [ ]:
symbols = list(port.get_available_symbols())[:7]
symbols

In [ ]:
ma = market_environment('ma', dt.date(2010, 1, 1))
ma.add_list('symbols', symbols)
ma.add_constant('source', 'google')  # not needed anymore
ma.add_constant('final date', dt.date(2014, 3, 1))

**Data retrieval** in this case takes a bit.

In [ ]:
%%time
port = mean_variance_portfolio('port', ma)
  # defining the portfolio and retrieving the data

In [ ]:
%%time
port.optimize('Vol')
print(port.variance, port.variance ** 0.5)
  # minimium variance & volatility in decimals

Given the larger data set now used, **efficient frontier** ...

In [ ]:
%%time
evols, erets = port.get_efficient_frontier(25)
  # efficient frontier of port

... and **capital market line** derivations take also longer.

In [ ]:
%%time
cml, optv, optr = port.get_capital_market_line(riskless_asset=0.01)
  # capital market line and optimal (tangent) portfolio

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(evols, erets, lw=2.0, label='efficient frontier')
plt.plot((0, 0.4), (cml(0), cml(0.4)), lw=2.0, label='capital market line')
plt.plot(optv, optr, 'r*', markersize=10, label='optimal portfolio')
plt.legend(loc=0)
plt.ylim(0)
plt.xlabel('expected volatility')
plt.ylabel('expected return');

**Copyright, License & Disclaimer**

© Dr. Yves J. Hilpisch | The Python Quants GmbH

DX Analytics (the "dx library" or "dx package") is licensed under the GNU Affero General
Public License version 3 or later (see http://www.gnu.org/licenses/).

DX Analytics comes with no representations or warranties, to the extent
permitted by applicable law.

http://tpq.io | [dx@tpq.io](mailto:team@tpq.io) |
http://twitter.com/dyjh

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

**Quant Platform** | http://pqp.io

**Python for Finance Training** | http://training.tpq.io

**Certificate in Computational Finance** | http://compfinance.tpq.io

**Derivatives Analytics with Python (Wiley Finance)** |
http://dawp.tpq.io

**Python for Finance (2nd ed., O'Reilly)** |
http://py4fi.tpq.io